# Plant Disease Detection — Final Project Submission (Jupyter Notebook)

This notebook trains and evaluates a CNN model to classify plant leaf images into: Healthy, Early Blight, Late Blight, and Bacterial Spot. It saves the trained model to `model/plant_disease_model.h5` and class labels to `class_names.txt`. If the PlantVillage dataset is not present, it will generate a small synthetic dataset to demonstrate the pipeline.

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

tf.get_logger().setLevel('ERROR')
CLASSES = ['Healthy', 'Early Blight', 'Late Blight', 'Bacterial Spot']
IMG_SIZE = (128, 128)
DATASET_DIR = os.path.join('PlantVillage', 'PlantVillage')
MODEL_DIR = 'model'
MODEL_PATH = os.path.join(MODEL_DIR, 'plant_disease_model.h5')
CLASS_NAMES_FILE = 'class_names.txt'
os.makedirs(MODEL_DIR, exist_ok=True)


## Data Loading
Uses `image_dataset_from_directory` if dataset is present. Otherwise creates a small synthetic dataset.

In [ ]:
def build_synthetic_dataset(base_dir, classes, n_per_class=40):
    os.makedirs(base_dir, exist_ok=True)
    for label in classes:
        cls_dir = os.path.join(base_dir, label)
        os.makedirs(cls_dir, exist_ok=True)
        for i in range(n_per_class):
            arr = np.zeros((IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.uint8)
            rng = np.random.default_rng(seed=i)
            color = np.array(rng.integers(50, 200, size=3), dtype=np.uint8)
            arr[:] = color
            noise = rng.integers(0, 40, size=arr.shape, dtype=np.uint8)
            arr = np.clip(arr + noise, 0, 255)
            Image.fromarray(arr).save(os.path.join(cls_dir, f'{label}_{i}.png'))

def get_datasets(dataset_dir, classes, img_size=(128,128), batch_size=32):
    if os.path.isdir(dataset_dir) and any(os.path.isdir(os.path.join(dataset_dir, d)) for d in os.listdir(dataset_dir)):
        train_ds = tf.keras.utils.image_dataset_from_directory(
            dataset_dir,
            validation_split=0.2,
            subset='training',
            seed=123,
            image_size=img_size,
            batch_size=batch_size
        )
        val_ds = tf.keras.utils.image_dataset_from_directory(
            dataset_dir,
            validation_split=0.2,
            subset='validation',
            seed=123,
            image_size=img_size,
            batch_size=batch_size
        )
    else:
        syn_dir = os.path.join('Dataset', 'Synthetic')
        build_synthetic_dataset(syn_dir, classes, n_per_class=60)
        train_ds = tf.keras.utils.image_dataset_from_directory(
            syn_dir,
            validation_split=0.2,
            subset='training',
            seed=123,
            image_size=img_size,
            batch_size=batch_size
        )
        val_ds = tf.keras.utils.image_dataset_from_directory(
            syn_dir,
            validation_split=0.2,
            subset='validation',
            seed=123,
            image_size=img_size,
            batch_size=batch_size
        )
    class_names = train_ds.class_names
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
    return train_ds, val_ds, class_names

train_ds, val_ds, detected_classes = get_datasets(DATASET_DIR, CLASSES, IMG_SIZE, batch_size=32)
detected_classes

## Model
CNN with 4 conv layers, max pooling, dropout, and dense softmax head.

In [ ]:
def build_model(input_shape=(128,128,3), num_classes=4):
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.Rescaling(1./255)(inputs)
    x = keras.layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Dropout(0.5)(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(128, activation='relu')(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

num_classes = len(detected_classes)
model = build_model((IMG_SIZE[0], IMG_SIZE[1], 3), num_classes)
model.summary()


## Training

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor='val_accuracy', save_best_only=True)
]
history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

with open(CLASS_NAMES_FILE, 'w') as f:
    for name in detected_classes:
        f.write(name + '
')

hist = history.history
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(hist['accuracy'], label='train')
plt.plot(hist['val_accuracy'], label='val')
plt.title('Accuracy')
plt.legend()
plt.subplot(1,2,2)
plt.plot(hist['loss'], label='train')
plt.plot(hist['val_loss'], label='val')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()


## Evaluation

In [ ]:
y_true = []
y_pred = []
for images, labels in val_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

print(classification_report(y_true, y_pred, target_names=detected_classes))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,5))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xticks(range(len(detected_classes)), detected_classes, rotation=45)
plt.yticks(range(len(detected_classes)), detected_classes)
plt.colorbar()
plt.tight_layout()
plt.show()


## Sample Inference

In [ ]:
def preprocess_image(path, target_size=(128,128)):
    img = Image.open(path).convert('RGB')
    img = img.resize(target_size)
    arr = np.array(img).astype('float32')/255.0
    arr = np.expand_dims(arr, axis=0)
    return arr

sample_path = None
for cls in detected_classes:
    cls_dir = os.path.join(DATASET_DIR, cls)
    if os.path.isdir(cls_dir):
        files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        if files:
            sample_path = os.path.join(cls_dir, files[0])
            break
if sample_path is None:
    syn_dir = os.path.join('Dataset','Synthetic', detected_classes[0])
    files = os.listdir(syn_dir)
    sample_path = os.path.join(syn_dir, files[0])

x = preprocess_image(sample_path, IMG_SIZE)
probs = model.predict(x, verbose=0)[0]
pred_idx = int(np.argmax(probs))
pred_label = detected_classes[pred_idx]
confidence = float(probs[pred_idx])
print(json.dumps({'predicted_label': pred_label, 'confidence': round(confidence*100,2)}, indent=2))


---
- Python: 3.8+
- TensorFlow/Keras: 2.15.0
- Dataset directory expected: `PlantVillage/PlantVillage/` (excluded from repo).